# Regional & Global Carbon Stocks in Mountains

Mountain (K1/K2/K3) carbon across Global, US/CAN, Canada, USA and Y2Y regions.
Merged from `calculate_{global,us_can}_carbon_stats_{k1,k2,k3}`. SOC is OLM-only; biomass/LC masking are the shared consistent versions.
k1/k2/k3 are stacked as rows (a `mask` column) within each region; values are flattened scalars. All exports are async `Export.table.toDrive`.


## Load packages and initialize GEE

In [23]:
# import packages
import ee
import geemap
import pandas as pd

In [24]:
# authenticate the EE api
ee.Authenticate()

True

In [25]:
# initialize the EE api
ee.Initialize(project='y2y-climate-benefits')

## Define GEE datasets

In [26]:
# define EE datasets
# biomass
biomass = ee.ImageCollection("projects/sat-io/open-datasets/ESA/ESA_CCI_AGB")
rsr = ee.Image("projects/y2y-climate-benefits/assets/inputs/Root_shoot_ratio_Map_Merged")

# soc
# unused: Sothe Canada SOC (global comparison uses OLM-only)
# soc_sothe = ee.Image(
#     "projects/y2y-climate-benefits/assets/inputs/McMaster_WWFCanada_soil_carbon1m_250m_kg-m2_version3_SCANLINE_ARTIFACT_FIX")
soc_0_10_olm_global = ee.Image(
    "projects/y2y-climate-benefits/assets/inputs/soc_0_10cm_kg_m2_olm_global")
soc_10_30_olm_global = ee.Image(
    "projects/y2y-climate-benefits/assets/inputs/soc_10_30cm_kg_m2_olm_global")
soc_30_60_olm_global = ee.Image(
    "projects/y2y-climate-benefits/assets/inputs/soc_30_60cm_kg_m2_olm_global")
soc_60_100_olm_global = ee.Image(
    "projects/y2y-climate-benefits/assets/inputs/soc_60_100cm_kg_m2_olm_global")

# peatlands
peat = ee.Image("projects/sat-io/open-datasets/GLOBAL-PEATLAND-DATABASE")

# landcover
lc = ee.Image("USGS/NLCD_RELEASES/2020_REL/NALCMS")
wte = ee.Image("projects/y2y-climate-benefits/assets/inputs/WTE_2020")
k1 = ee.Image("projects/y2y-climate-benefits/assets/inputs/k1_binary")
k2 = ee.Image("projects/y2y-climate-benefits/assets/inputs/k2_binary").unmask()
k3 = ee.Image("projects/y2y-climate-benefits/assets/inputs/k3_binary").unmask()

# global landcover 2022
lc_cci = (
    ee.ImageCollection("projects/sat-io/open-datasets/ESA/C3S-LC-L4-LCCS")
    .filter(ee.Filter.stringContains("system:index", "2022")).first() #2022
)

# dem mask for land surface
earth_mask = ee.FeatureCollection("projects/y2y-climate-benefits/assets/inputs/land_polygons")

# netflux
netflux = ee.ImageCollection(
    "projects/wri-datalab/gfw-data-lake/net-flux-forest-extent-per-ha-v1-3-2-2001-2023/net-flux-global-forest-extent-per-ha-2001-2023"
)

# irrecoverable carbon (Noon layers)
irr_carbon_biomass = ee.Image("projects/sat-io/open-datasets/irrecoverable_carbon/carbon_biomass/carbon_biomass_2018")
irr_carbon_biomass_se = ee.Image("projects/y2y-climate-benefits/assets/inputs/Irrecoverable_Carbon_Biomass_Uncertainty_20210329")
irr_carbon_soc = ee.Image("projects/sat-io/open-datasets/irrecoverable_carbon/carbon_soil/carbon_soil_2018")
irr_carbon_soc_se = ee.Image("projects/y2y-climate-benefits/assets/inputs/Irrecoverable_Carbon_Soil_Uncertainty_20210329")
irr_carbon_total = ee.Image("projects/sat-io/open-datasets/irrecoverable_carbon/carbon_total/carbon_total_2018")
irr_carbon_total_se = ee.Image("projects/y2y-climate-benefits/assets/inputs/Irrecoverable_Carbon_Uncertainty_20210329")

# vector
y2y = ee.FeatureCollection("projects/y2y-climate-benefits/assets/inputs/y2y")
y2y_ecoregions = ee.FeatureCollection(
    "projects/y2y-climate-benefits/assets/inputs/y2y_ecoregions")
y2y_biomes = ee.FeatureCollection("projects/y2y-climate-benefits/assets/inputs/y2y_biomes")
rra = ee.FeatureCollection("projects/y2y-climate-benefits/assets/inputs/ross_river_ipca")
countries = ee.FeatureCollection("USDOS/LSIB/2017")
us_can = ee.FeatureCollection("projects/y2y-climate-benefits/assets/inputs/us_can_simple")

# carbon deficit
agb_mean_act = ee.Image('projects/y2y-climate-benefits/assets/inputs/tcd_agbc_mean_act')
agb_mean_prim = ee.Image('projects/y2y-climate-benefits/assets/inputs/tcd_agbc_mean_prim')
bgb_mean_act = ee.Image('projects/y2y-climate-benefits/assets/inputs/tcd_bgbc_mean_act')
bgb_mean_prim = ee.Image('projects/y2y-climate-benefits/assets/inputs/tcd_bgbc_mean_prim')
soc_mean_act = ee.Image('projects/y2y-climate-benefits/assets/inputs/tcd_soc_mean_act')
soc_mean_prim = ee.Image('projects/y2y-climate-benefits/assets/inputs/tcd_soc_mean_prim')

## Define regions


In [27]:
# global geometry (excluding antarctica)
globe = ee.Geometry.Rectangle(coords=[-180, -60, 180, 90], proj=ee.Projection('EPSG:4326'), geodesic=False)

# geometry bigger than us/canada
big_geo = ee.Geometry.Rectangle(coords=[171, 18, 308, 84], proj=ee.Projection('EPSG:4326'), geodesic=False)

# country subsets
canada = countries.filter(ee.Filter.eq('COUNTRY_NA', 'Canada'))
usa = countries.filter(ee.Filter.eq('COUNTRY_NA', 'United States'))

## Calculate biomass carbon


In [28]:
# calc ESA CCI biomass
# grab 2022 AGB images
agb = biomass.filter(ee.Filter.stringContains("system:index", "2022")).first().select(['AGB'])
agb_sd = biomass.filter(ee.Filter.stringContains("system:index", "2022")).first().select(['SD'])

# create image for litter using Harris ratio (4%)
# leave out dead wood since it is likely sensed already
# using global LC raster now
# mask litter only for forested LC types
#   '006400', // 50: Tree cover, broadleaved, evergreen, closed to open (>15%)
#   '00a000', // 60: Tree cover, broadleaved, deciduous, closed to open (>15%)
#   '003c00', // 70: Tree cover, needleleaved, evergreen, closed to open (>15%)
#   '285000', // 80: Tree cover, needleleaved, deciduous, closed to open (>15%)
#   '788200', // 90: Tree cover, mixed leaf type
#   '8ca000', // 100: Mosaic tree and shrub (>50%) / herbaceous cover (<50%)
#   '00785a', // 160: Tree cover, flooded, fresh or brackish water
#   '009678', // 170: Tree cover, flooded, saline water
#   '0046c8', // 210: Water bodies
#   'ffffff'  // 220: Permanent snow and ice

litter_mask = (
    lc_cci.eq(50)
    .Or(lc_cci.eq(60))
    .Or(lc_cci.eq(70))
    .Or(lc_cci.eq(80))
    .Or(lc_cci.eq(90))
    .Or(lc_cci.eq(100))
    .Or(lc_cci.eq(160))
    .Or(lc_cci.eq(170))
)

litter = agb.multiply(0.04).multiply(litter_mask)
litter_sd = agb_sd.multiply(0.04).multiply(litter_mask)

# create image for BGB (using global rsr map)
bgb = agb.multiply(rsr).rename('BGB')
bgb_sd = agb_sd.multiply(rsr).rename('BGB_SD')

# add AGB + litter + BGB together
bio = agb.add(litter).addBands(
    bgb).rename(['agb_t_ha', 'bgb_t_ha'])
bio_se = agb_sd.add(litter_sd).addBands( # add linearly since not independent
    bgb_sd).rename(['agb_sd_t_ha', 'bgb_sd_t_ha']) 

# multiply values by 0.47 to get carbon density
# 0.47 used by Harris et al. (2021)
# mask water (210) and snow/ice (220)
bio = bio.multiply(0.47).updateMask(
    lc_cci.neq(210).And(lc_cci.neq(220)))
bio_se = bio_se.multiply(0.47).updateMask(
    lc_cci.neq(210).And(lc_cci.neq(220)))

# compute per-pixel area in ha
pixel_area_ha = ee.Image.pixelArea().divide(10000)


## Calculate soil organic carbon (OLM-only)


In [29]:
# calc open land map soc
# add the layers together from 0-100cm
soc_olm = (
    soc_0_10_olm_global
    .add(soc_10_30_olm_global)
    .add(soc_30_60_olm_global)
    .add(soc_60_100_olm_global)
    )

# multiply by 10 to get t/ha
# mask water/snow/ice
soc_olm = soc_olm.multiply(10).rename(
    'soc_dens').updateMask(lc_cci.neq(210).And(lc_cci.neq(220)))


In [30]:
# OLM-only SOC for the global comparison (no Sothe Canada blend)
soc_blend = soc_olm.rename('soc_t_ha').reproject(
    crs=soc_0_10_olm_global.projection(),
    crsTransform=soc_0_10_olm_global.projection().getInfo().get('transform')
).updateMask(lc_cci.neq(210).And(lc_cci.neq(220)))  # mask snow/ice, water

# Create soc layer mask to filter pixel area raster
soc_mask = soc_blend.mask().neq(0)

# Mask pixel_area_ha to carbon layers
pixel_area_soc_extent = pixel_area_ha.updateMask(soc_mask)

# multiply by pixel area to get total carbon per pixel
soc_blend_stock = soc_blend.multiply(pixel_area_ha).rename('soc_t').addBands(
    pixel_area_soc_extent.rename(['pixel_area_soc_extent_ha'])
).addBands(
    pixel_area_ha.rename(['pixel_area_ha'])
)

## Helper functions (k1/k2/k3 stacked; flattened scalars)


In [31]:
MAXPIX = 1e20
masks = [('k1', k1), ('k2', k2), ('k3', k3)]

def _val(d, factor=1):
    # flatten a single-band reduceRegion dict to a scalar (None if empty),
    # dividing by `factor` for unit conversion (branch on size so 0 survives)
    d = ee.Dictionary(d)
    return ee.Algorithms.If(d.size().gt(0),
                            ee.Number(d.values().get(0)).divide(factor), None)

def _sum(img, geom, scale_img, factor=1e6):
    # sum of (t/ha * ha) = tonnes; /1e6 -> MtC
    return _val(img.reduceRegion(reducer=ee.Reducer.sum(), geometry=geom,
                                 scale=scale_img.projection().nominalScale(), maxPixels=MAXPIX), factor)

def export_fc(features, description):
    ee.batch.Export.table.toDrive(collection=ee.FeatureCollection(features),
                                  description=description, folder='', fileFormat='CSV').start()

def carbon_stock_feature(mask_name, mask, clip, geom, total=False):
    def prep(img):
        i = img.reduce(ee.Reducer.sum())
        if clip is not None:
            i = i.clipToCollection(clip)
        return i.multiply(pixel_area_ha)
    def socprep():
        i = soc_blend
        if clip is not None:
            i = i.clipToCollection(clip)
        return i.multiply(pixel_area_ha)
    bp, bsp, sp = prep(bio), prep(bio_se), socprep()
    props = {
        'mask': mask_name,
        'biomass_MtC':        _sum(bp.updateMask(mask), geom, bio),
        'biomass_not_MtC':    _sum(bp.updateMask(mask.Not()), geom, bio),
        'biomass_se_MtC':     _sum(bsp.updateMask(mask), geom, bio_se),
        'biomass_se_not_MtC': _sum(bsp.updateMask(mask.Not()), geom, bio_se),
        'soc_MtC':            _sum(sp.updateMask(mask), geom, soc_blend),
        'soc_not_MtC':        _sum(sp.updateMask(mask.Not()), geom, soc_blend),
    }
    if total:
        props['biomass_total_MtC'] = _sum(bp, geom, bio)
        props['soc_total_MtC']     = _sum(sp, geom, soc_blend)
    return ee.Feature(None, props)

def noon_feature(mask_name, mask, clip, geom, types, y2y_slice=False, region_total=False):
    def prep(img):
        i = img
        if clip is not None:
            i = i.clipToCollection(clip)
        return i.multiply(pixel_area_ha)
    props = {'mask': mask_name}
    for nm, img, img_se in types:
        p, pse = prep(img), prep(img_se)
        props[nm + '_MtC']           = _sum(p.updateMask(mask), geom, img)
        props[nm + '_not_MtC']  = _sum(p.updateMask(mask.Not()), geom, img)
        props[nm + '_se_MtC']   = _sum(pse.updateMask(mask), geom, img_se)
        props[nm + '_se_not_MtC'] = _sum(pse.updateMask(mask.Not()), geom, img_se)
        if region_total:
            props[nm + '_total_MtC'] = _sum(p, geom, img)
        if y2y_slice:
            props[nm + '_y2y_MtC']    = _sum(img.multiply(pixel_area_ha), y2y.geometry(), img)
            props[nm + '_se_y2y_MtC'] = _sum(img_se.multiply(pixel_area_ha), y2y.geometry(), img_se)
    return ee.Feature(None, props)

def area_feature(region, mask_name, mask, clip, geom, total=False):
    a = pixel_area_ha
    if clip is not None:
        a = a.clipToCollection(clip)
    props = {
        'region': region,
        'mask': mask_name,
        'area_Mha':     _val(a.updateMask(mask).reduceRegion(ee.Reducer.sum(), geom, scale=100, maxPixels=MAXPIX), 1e6),
        'area_Mha_not': _val(a.updateMask(mask.Not()).reduceRegion(ee.Reducer.sum(), geom, scale=100, maxPixels=MAXPIX), 1e6),
    }
    if total:
        props['area_Mha_total'] = _val(a.reduceRegion(ee.Reducer.sum(), geom, scale=100, maxPixels=MAXPIX), 1e6)
    return ee.Feature(None, props)

def deficit_feature(mask_name, mask):
    layers = [('agb_act', agb_mean_act), ('agb_prim', agb_mean_prim),
              ('bgb_act', bgb_mean_act), ('bgb_prim', bgb_mean_prim),
              ('soc_act', soc_mean_act), ('soc_prim', soc_mean_prim)]
    props = {'mask': mask_name}
    for nm, lyr in layers:
        p = lyr.multiply(pixel_area_ha)
        props[nm + '_MtC']          = _sum(p.updateMask(mask), globe, lyr)
        props[nm + '_not_MtC'] = _sum(p.updateMask(mask.Not()), globe, lyr)
    return ee.Feature(None, props)

NOON_TYPES_2 = [('irr_carbon_biomass', irr_carbon_biomass, irr_carbon_biomass_se),
                ('irr_carbon_soc', irr_carbon_soc, irr_carbon_soc_se)]
NOON_TYPES_3 = NOON_TYPES_2 + [('irr_carbon_total', irr_carbon_total, irr_carbon_total_se)]

## Carbon stock stats


In [32]:
export_fc([carbon_stock_feature(mn, m, None, globe) for mn, m in masks],
          'carbon_stock_mountain_stats_global_olm')

In [33]:
export_fc([carbon_stock_feature(mn, m, us_can, big_geo) for mn, m in masks],
          'carbon_stock_mountain_stats_us_can_olm')

In [34]:
export_fc([carbon_stock_feature(mn, m, canada, big_geo) for mn, m in masks],
          'carbon_stock_mountain_stats_canada_olm')

In [35]:
# USA is k1-only in the original analysis; includes region totals
export_fc([carbon_stock_feature('k1', k1, usa, big_geo, total=True)],
          'carbon_stock_mountain_stats_USA_olm')

In [36]:
export_fc([carbon_stock_feature(mn, m, None, y2y.geometry()) for mn, m in masks],
          'carbon_stock_mountain_stats_y2y_olm')

## Irrecoverable carbon stats (Noon layers)


In [37]:
# global includes total carbon type and the y2y sub-slice
export_fc([noon_feature(mn, m, None, globe, NOON_TYPES_3, y2y_slice=True) for mn, m in masks],
          'noon_global_irr_carbon_stats')

In [38]:
export_fc([noon_feature(mn, m, us_can, big_geo, NOON_TYPES_2) for mn, m in masks],
          'noon_us_can_irr_carbon_stats')

In [39]:
export_fc([noon_feature(mn, m, canada, big_geo, NOON_TYPES_2) for mn, m in masks],
          'noon_can_irr_carbon_stats')

In [40]:
# USA is k1-only; includes region totals
export_fc([noon_feature('k1', k1, usa, big_geo, NOON_TYPES_2, region_total=True)],
          'noon_usa_irr_carbon_stats')

In [41]:
export_fc([noon_feature(mn, m, None, y2y.geometry(), NOON_TYPES_2) for mn, m in masks],
          'noon_y2y_irr_carbon_stats')

In [42]:
# Y2Y-Canada is k1-only
export_fc([noon_feature('k1', k1, canada, y2y.geometry(), NOON_TYPES_2)],
          'noon_y2y_can_irr_carbon_stats')

## Carbon deficit stats (Global, K1 only)


In [43]:
export_fc([deficit_feature('k1', k1)], 'global_carbon_deficit_stats')

## Area stats


In [44]:
# all area stats -> one file (region + mask columns)
# global is land-masked via earth_mask; y2y uses the polygon directly (no clip)
area_rows = (
    [area_feature('global', mn, m, earth_mask, globe) for mn, m in masks]
    + [area_feature('us_can', mn, m, us_can, big_geo) for mn, m in masks]
    + [area_feature('canada', mn, m, canada, big_geo) for mn, m in masks]
    + [area_feature('usa', mn, m, usa, big_geo) for mn, m in masks]
    + [area_feature('y2y', mn, m, None, y2y.geometry()) for mn, m in masks]
)
export_fc(area_rows, 'mountain_area_stats')